# ConvTran: Positional Encoding Experiments

We study how the positional encoding affects ConvTran when the convolutional stem stays the same. For each variant, we record test accuracy, the time per training epoch and the peak CUDA memory during training.

We compare three variants built on the released implementation. The `upstream/ConvTran` submodule is used at its pinned commit and is not modified.

| Variant | Absolute encoding | Attention |
|---|---|---|
| `tape_erpe` | tAPE | eRPE (`Attention_Rel_Scl`) |
| `sinusoidal` | Sinusoidal (`AbsolutePositionalEncoding`) | Dense `Attention`, eRPE disabled |
| `rope` | None | Dense `Attention` with rotary encoding of Q and K, eRPE disabled |

`tape_erpe` is the released ConvTran configuration, and `sinusoidal` uses an absolute encoding already defined in the upstream code. RoPE is not part of the upstream code; we add it in `scripts/positional_encodings.py`.

In the `rope` variant, Q and K are rotated by position after they are split into heads and before the dot product. The values, the `emb_size ** -0.5` scale, the softmax and the output LayerNorm are those of the upstream dense attention. With an embedding dimension of 16 and 8 heads, each head has two coordinates, so a single pair is rotated by one radian per position.

All variants share the convolutional stem, the upstream loader and its cached preprocessing, and the fixed stratified validation split (`random_state=1234`). They also share the RAdam optimizer and learning rate, and the evaluation on the official test split of the checkpoint with the lowest validation loss. Synthetic checks of the variant selection and the RoPE dimensions are in `tests/test_positional_encodings.py`.

## Configuration

The notebook has two modes. The pilot trains each variant for two epochs with seed 1234. We have already completed it on the GPU for the three variants and keep it to check the pipeline and the measurements. The comparison trains each variant for 100 epochs with the five seeds used in the baseline.

| Setting | Pilot | Comparison |
|---|---|---|
| Dataset | BasicMotions | BasicMotions |
| Seeds | 1234 | 1234, 2345, 3456, 4567, 5678 |
| Epochs | 2 | 100 |
| Batch size | 16 | 16 |
| Embedding dimension | 16 | 16 |
| Attention heads | 8 | 8 |

`MODE` selects the mode and is set to the comparison. Training starts only when `RUN_TRAINING` is set to `True`; otherwise, Run All reads the saved results. The cell below also checks the runner, the submodule commit and the dataset files.

In [5]:
from pathlib import Path
import shlex
import subprocess
import sys

import pandas as pd

MODE = "comparison"  # "pilot" or "comparison"
RUN_TRAINING = True  # Set to True, then Run All, to train the selected mode.

MODES = {
    "pilot": {"seeds": [1234], "epochs": 2},
    "comparison": {"seeds": [1234, 2345, 3456, 4567, 5678], "epochs": 100},
}
DATASET = "Handwriting"
VARIANTS = ["tape_erpe", "sinusoidal", "rope"]
BATCH_SIZE = 16
EMB_SIZE = 16
NUM_HEADS = 8
assert MODE in MODES, f"MODE must be one of {sorted(MODES)}"
SEEDS = MODES[MODE]["seeds"]
EPOCHS = MODES[MODE]["epochs"]

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)
assert ROOT is not None, "Open this notebook from the project repository."

RUNNER = ROOT / "scripts" / "run_positional_experiment.py"
UPSTREAM = ROOT / "upstream" / "ConvTran"
DATA_DIR = ROOT / "Dataset" / "UEA" / "Multivariate_ts" / DATASET

assert RUNNER.is_file(), f"Missing experiment runner: {RUNNER}"
for split in ("TRAIN", "TEST"):
    path = DATA_DIR / f"{DATASET}_{split}.ts"
    assert path.is_file(), f"Missing dataset file: {path}"

pinned = subprocess.check_output(
    ["git", "-C", str(ROOT), "ls-tree", "HEAD", "upstream/ConvTran"], text=True
).split()[2]
checked_out = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
).strip()
upstream_changes = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "status", "--porcelain"], text=True
).strip()
assert checked_out == pinned, f"Submodule at {checked_out}, pinned {pinned}"
assert not upstream_changes, "The upstream submodule has local changes."

sys.path.insert(0, str(ROOT / "scripts"))
from run_positional_experiment import FIELDS, RESULTS_PATH, run_directory

def run_dir(variant, seed):
    return run_directory(
        variant, DATASET, seed, EPOCHS, BATCH_SIZE, EMB_SIZE, NUM_HEADS
    )

def runner_args(variant, seed):
    return [
        "--variant", variant,
        "--dataset", DATASET,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--emb-size", str(EMB_SIZE),
        "--num-heads", str(NUM_HEADS),
    ]

def retry_command(variant, seed):
    python = Path(sys.executable)
    if python.is_relative_to(ROOT):
        python = python.relative_to(ROOT)
    return shlex.join([
        str(python), str(RUNNER.relative_to(ROOT)),
        *runner_args(variant, seed), "--retry",
    ])

def as_available(table):
    return table.astype(object).where(table.notna(), "not available")

print("Repository:", ROOT)
print("Python:", sys.executable)
print("Upstream commit (pinned, clean):", pinned)
print("Results CSV:", RESULTS_PATH.relative_to(ROOT))
print(f"Mode: {MODE} ({EPOCHS} epochs, seeds {SEEDS})")
print("RUN_TRAINING:", RUN_TRAINING)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python
Upstream commit (pinned, clean): 148afb6ca549915b7e78b05e2ec3b4ba6e052341
Results CSV: results/positional_encoding_runs.csv
Mode: comparison (100 epochs, seeds [1234, 2345, 3456, 4567, 5678])
RUN_TRAINING: True


## Training

When training is enabled, the cell below runs every combination of variant and seed in the selected mode, one after another. The comparison mode gives 15 runs. Each run calls `scripts/run_positional_experiment.py`, which starts the upstream `main.py` in a separate process and writes its own log.

Logs, checkpoints and configurations are saved under `outputs/experiments/positional_encoding/<variant>/<dataset>/seed<seed>_ep<epochs>_bs<batch>_emb<emb>_h<heads>/`. Completed runs are added to `results/positional_encoding_runs.csv`. The runner reuses a run only if its log for the same configuration is complete.

If a run fails, the cell prints the error and the log path and continues with the next run. The incomplete log is kept. To try again, run the `--retry` command printed for that run; it renames the old log to `run.failed-<UTC time>.log` before training.

In [6]:
planned = [(variant, seed) for seed in SEEDS for variant in VARIANTS]
commands = [
    [sys.executable, str(RUNNER), *runner_args(variant, seed)]
    for variant, seed in planned
]
print(
    f"{MODE}: {len(commands)} runs planned "
    f"({len(VARIANTS)} variants x {len(SEEDS)} seeds, {EPOCHS} epochs)"
)

attempts = []
if not RUN_TRAINING:
    print("Training disabled; set RUN_TRAINING = True in the configuration cell.")

for (variant, seed), command in zip(planned, commands) if RUN_TRAINING else []:
    log = run_dir(variant, seed) / "run.log"
    print(f"\n[{variant}, seed {seed}] log: {log.relative_to(ROOT)}", flush=True)
    process = subprocess.run(
        command, cwd=ROOT, capture_output=True, text=True, check=False
    )
    if process.stdout.strip():
        print(process.stdout.strip())
    if process.returncode:
        print(f"[{variant}, seed {seed}] FAILED with exit code {process.returncode}")
        print(process.stderr.strip())
        print(f"Log: {log}")
        print(f"Retry: {retry_command(variant, seed)}")
    attempts.append({
        "variant": variant,
        "seed": seed,
        "exit_code": process.returncode,
        "log": str(log.relative_to(ROOT)),
    })

if attempts:
    display(pd.DataFrame(attempts))

comparison: 15 runs planned (3 variants x 5 seeds, 100 epochs)

[tape_erpe, seed 1234] log: outputs/experiments/positional_encoding/tape_erpe/handwriting/seed1234_ep100_bs16_emb16_h8/run.log


Training tape_erpe on Handwriting. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/positional_encoding/tape_erpe/handwriting/seed1234_ep100_bs16_emb16_h8/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/positional_encoding_runs.csv

[sinusoidal, seed 1234] log: outputs/experiments/positional_encoding/sinusoidal/handwriting/seed1234_ep100_bs16_emb16_h8/run.log
Training sinusoidal on Handwriting. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/positional_encoding/sinusoidal/handwriting/seed1234_ep100_bs16_emb16_h8/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/positional_encoding_runs.csv

[rope, seed 1234] log: outputs/experiments/positional_encoding/rope/handwriting/seed1234_ep100_bs16_emb16_h8/run.log
Training rope on Handwriting. Log: /home/tyrerodr/Documents/Msc-Adva

,variant,seed,exit_code,log
0,tape_erpe,1234,0,outputs/experiments/positional_encoding/tape_e...
1,sinusoidal,1234,0,outputs/experiments/positional_encoding/sinuso...
2,rope,1234,0,outputs/experiments/positional_encoding/rope/h...
3,tape_erpe,2345,0,outputs/experiments/positional_encoding/tape_e...
4,sinusoidal,2345,0,outputs/experiments/positional_encoding/sinuso...
5,rope,2345,0,outputs/experiments/positional_encoding/rope/h...
6,tape_erpe,3456,0,outputs/experiments/positional_encoding/tape_e...
7,sinusoidal,3456,0,outputs/experiments/positional_encoding/sinuso...
8,rope,3456,0,outputs/experiments/positional_encoding/rope/h...
9,tape_erpe,4567,0,outputs/experiments/positional_encoding/tape_e...


## Results by run

The table has one row for each expected combination of variant and seed in the selected mode. It reads `results/positional_encoding_runs.csv` and the run directories, so it also works when training is disabled. Rows from the other mode are excluded, and missing values are shown as not available.

A run is complete when the CSV has a row with the same variant, seed, dataset, number of epochs, batch size, embedding dimension and number of heads. A run is marked as incomplete or failed when its log exists without a recorded result.

Test accuracy and test loss come from the upstream evaluation. After training, the checkpoint with the lowest validation loss is loaded and evaluated on the official test split.

Epoch times measure each call to the upstream `train_epoch`, with CUDA synchronized at the start and end. They include loading batches, the forward and backward passes, gradient clipping and the optimizer step. Validation, checkpoint saving, logging and the test evaluation are excluded. We report the first epoch separately because it includes one-time costs, such as CUDA and cuDNN initialization and the creation of the optimizer state. The later-epoch time is the mean of epochs 2 to the last epoch.

Peak memory is `torch.cuda.max_memory_allocated`. The counter is reset once the model is on the GPU, at the start of the upstream `train_runner`, and read when it returns, before testing. Upstream validates after every training epoch, so the value covers both training and validation. It counts tensors allocated by PyTorch, not memory reserved by its allocator, the CUDA context or other processes, so it is lower than the value reported by `nvidia-smi`. It is not available for runs on the CPU.

In [7]:
results = (
    pd.read_csv(RESULTS_PATH) if RESULTS_PATH.is_file()
    else pd.DataFrame(columns=FIELDS)
)
selected = results[
    (results["dataset"] == DATASET)
    & (results["epochs"] == EPOCHS)
    & (results["batch_size"] == BATCH_SIZE)
    & (results["emb_size"] == EMB_SIZE)
    & (results["num_heads"] == NUM_HEADS)
    & results["variant"].isin(VARIANTS)
    & results["seed"].isin(SEEDS)
]

def epoch_times(value):
    if pd.isna(value):
        return []
    return [float(part) for part in str(value).split(";") if part]

rows = []
pending_retries = []
for variant in VARIANTS:
    for seed in SEEDS:
        match = selected[(selected["variant"] == variant) & (selected["seed"] == seed)]
        log = run_dir(variant, seed) / "run.log"
        row = {"variant": variant, "seed": seed, "log": str(log.relative_to(ROOT))}
        if not match.empty:
            record = match.iloc[-1]
            times = epoch_times(record["train_epoch_times_s"])
            row.update(
                status="complete",
                test_accuracy_pct=record["accuracy"] * 100,
                test_loss=record["test_loss"],
                first_epoch_s=times[0] if times else None,
                later_epochs_mean_s=(
                    sum(times[1:]) / len(times[1:]) if len(times) > 1 else None
                ),
                timed_epochs=len(times),
                peak_cuda_alloc_train_val_mib=record["peak_cuda_allocated_train_val_mib"],
                device=record["device"],
            )
        elif log.is_file():
            row["status"] = "incomplete or failed"
            pending_retries.append(retry_command(variant, seed))
        else:
            row["status"] = "not run"
        rows.append(row)

RUN_COLUMNS = [
    "variant", "seed", "status", "test_accuracy_pct", "test_loss",
    "first_epoch_s", "later_epochs_mean_s", "timed_epochs",
    "peak_cuda_alloc_train_val_mib", "device", "log",
]
runs_table = pd.DataFrame(rows).reindex(columns=RUN_COLUMNS)
for column in RUN_COLUMNS[3:7] + ["peak_cuda_alloc_train_val_mib"]:
    runs_table[column] = pd.to_numeric(runs_table[column])
runs_table["timed_epochs"] = runs_table["timed_epochs"].astype("Int64")

print(f"{MODE}: {len(selected)} of {len(rows)} expected runs recorded")
display(as_available(runs_table.round(4)))
if pending_retries:
    print("Incomplete logs are kept. Inspect each log, then retry from the repository root:")
    for command in pending_retries:
        print(" ", command)

comparison: 15 of 15 expected runs recorded


,variant,seed,status,test_accuracy_pct,test_loss,first_epoch_s,later_epochs_mean_s,timed_epochs,peak_cuda_alloc_train_val_mib,device,log
0,tape_erpe,1234,complete,21.6471,2.8509,0.206,0.0173,100,71.0161,cuda,outputs/experiments/positional_encoding/tape_e...
1,tape_erpe,2345,complete,22.7059,2.7171,0.1944,0.0171,100,71.0161,cuda,outputs/experiments/positional_encoding/tape_e...
2,tape_erpe,3456,complete,25.7647,2.6719,0.195,0.0174,100,71.0161,cuda,outputs/experiments/positional_encoding/tape_e...
3,tape_erpe,4567,complete,26.0,2.6748,0.1947,0.0174,100,71.0161,cuda,outputs/experiments/positional_encoding/tape_e...
4,tape_erpe,5678,complete,23.4118,2.5727,0.1934,0.0173,100,71.0161,cuda,outputs/experiments/positional_encoding/tape_e...
5,sinusoidal,1234,complete,25.7647,2.7481,0.1917,0.0166,100,71.2402,cuda,outputs/experiments/positional_encoding/sinuso...
6,sinusoidal,2345,complete,18.7059,2.7631,0.1936,0.0161,100,71.2402,cuda,outputs/experiments/positional_encoding/sinuso...
7,sinusoidal,3456,complete,28.1176,2.4814,0.1938,0.0164,100,71.2402,cuda,outputs/experiments/positional_encoding/sinuso...
8,sinusoidal,4567,complete,29.6471,2.5015,0.1932,0.0167,100,71.2402,cuda,outputs/experiments/positional_encoding/sinuso...
9,sinusoidal,5678,complete,22.1176,2.6229,0.1933,0.0163,100,71.2402,cuda,outputs/experiments/positional_encoding/sinuso...


## Summary by variant

We summarize the completed seeds of each variant. Accuracy is reported as the mean in percent and the sample standard deviation in percentage points. The standard deviation is not available with a single seed.

For epoch times, we first average epochs 2 to the last epoch within each seed and then average across seeds. The first-epoch time and the peak memory are also averaged across seeds.

A summary with pending seeds is labelled as partial. Partial summaries should not be compared directly with complete ones.

In [8]:
summary_rows = []
for variant in VARIANTS:
    done = runs_table[
        (runs_table["variant"] == variant) & (runs_table["status"] == "complete")
    ]
    completed = len(done)
    summary_rows.append({
        "variant": variant,
        "summary": "complete" if completed == len(SEEDS) else "partial",
        "seeds_completed": f"{completed}/{len(SEEDS)}",
        "mean_accuracy_pct": done["test_accuracy_pct"].mean(),
        "std_accuracy_pp": (
            done["test_accuracy_pct"].std(ddof=1) if completed > 1 else None
        ),
        "mean_first_epoch_s": done["first_epoch_s"].mean(),
        "mean_later_epochs_s": done["later_epochs_mean_s"].mean(),
        "mean_peak_cuda_alloc_train_val_mib": (
            done["peak_cuda_alloc_train_val_mib"].mean()
        ),
    })

summary = pd.DataFrame(summary_rows)
numeric = summary.columns[3:]
summary[numeric] = summary[numeric].apply(pd.to_numeric).round(4)
display(as_available(summary))

,variant,summary,seeds_completed,mean_accuracy_pct,std_accuracy_pp,mean_first_epoch_s,mean_later_epochs_s,mean_peak_cuda_alloc_train_val_mib
0,tape_erpe,complete,5/5,23.9059,1.9123,0.1967,0.0173,71.0161
1,sinusoidal,complete,5/5,24.8706,4.4649,0.1931,0.0164,71.2402
2,rope,complete,5/5,25.2471,1.642,0.2132,0.0188,71.1958


## Limitations

All results come from BasicMotions, which has 40 test samples, and the five seeds share one validation split. Each timing comes from a single run on a desktop GPU.

With one rotated pair per head, the RoPE angle grows by one radian per position and repeats roughly every 6.3 positions. This variant therefore gives only a coarse encoding of relative distance, and the comparison applies to this configuration rather than to RoPE in general.

The two-epoch pilot checks that the pipeline and the measurements work. It is too short to compare accuracy.